# Retrieval: hybrid search + reranking

## Setup

Importing the retrieval stack and connecting to Qdrant, checking that the indexed collection exists and is populated before querying

In [ ]:
import torch
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer, CrossEncoder

QDRANT_URL = "http://localhost:6333"
COLLECTION = "lecture_chunks"

DENSE_MODEL = "BAAI/bge-m3"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)

client = QdrantClient(url=QDRANT_URL)
print("Collections:", [c.name for c in client.get_collections().collections])
print("Punkte in", COLLECTION, ":", client.count(COLLECTION).count)

Loading the dense (BGE-M3) and sparse (BM25) encoders, the same models used at indexing time, so query and document vectors stay comparable

In [ ]:
dense_embedder = SentenceTransformer(DENSE_MODEL, device=DEVICE)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL, language="german")
print("Dense + Sparse geladen.")

## Embed one query

Encoding one example query with both encoders and inspecting the vectors, just to make the dense and sparse query representations concrete

In [ ]:
query = "Wie funktioniert eine Faltung und warum ist das relevant?"

q_dense = dense_embedder.encode(query, normalize_embeddings=True)
q_sparse = list(sparse_embedder.query_embed(query))[0]

print("Dense-Vektor:", q_dense.shape, q_dense.dtype)
print("Sparse-Tokens:", len(q_sparse.indices), "Non-Zero")
print("Erste Token-IDs: ", q_sparse.indices[:8])
print("Erste BM25-Werte:", q_sparse.values[:8])

## Dense only and sparse only search

In [ ]:
def dense_only(query, limit=5):
    q = dense_embedder.encode(query, normalize_embeddings=True)
    return client.query_points(
        collection_name=COLLECTION, query=q.tolist(), using="dense",
        limit=limit, with_payload=True,
    ).points

for i, r in enumerate(dense_only(query), start=1):
    print(f"[{i}] {r.score:.4f} | {r.payload['lecture']} p.{r.payload['page_numbers']} | {r.payload['title']}")

In [ ]:
def sparse_only(query, limit=5):
    q = list(sparse_embedder.query_embed(query))[0]
    return client.query_points(
        collection_name=COLLECTION,
        query=models.SparseVector(indices=q.indices.tolist(), values=q.values.tolist()),
        using="sparse", limit=limit, with_payload=True,
    ).points

for i, r in enumerate(sparse_only(query), 1):
    print(f"[{i}] {r.score:.4f} | {r.payload['lecture']} p.{r.payload['page_numbers']} | {r.payload['title']}")

## Hybrid search (RRF)

hybrid_search runs the dense and sparse prefetches and fuses the two rankings with RRF

In [ ]:
def hybrid_search(query, prefetch_limit=100):
    q_dense = dense_embedder.encode(query, normalize_embeddings=True)
    q_sparse = list(sparse_embedder.query_embed(query))[0]
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=q_dense.tolist(), using="dense", limit=prefetch_limit),
            models.Prefetch(
                query=models.SparseVector(indices=q_sparse.indices.tolist(), values=q_sparse.values.tolist()),
                using="sparse", limit=prefetch_limit,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=prefetch_limit, with_payload=True,
    ).points

candidates = hybrid_search(query, prefetch_limit=150)
print(f"{len(candidates)} Kandidaten. Top-10 nach RRF:\n")
for i, h in enumerate(candidates[:10], 1):
    print(f"[{i:2}] {h.score:.4f} | {h.payload['lecture']} p.{h.payload['page_numbers']} | {h.payload['title']}")

## Cross-encoder reranking

Loading the BGE cross-encoder reranker in fp16

In [ ]:
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)
if DEVICE == "cuda":
    reranker.model.half()  # fp16 only pays off on NVIDIA GPUs
print("Reranker geladen:", RERANK_MODEL, "|", "fp16" if DEVICE == "cuda" else DEVICE)

## Reranking: top-20 to top-5

Scoring each [query, passage] pair with the cross-encoder and re-sorting the candidates, keeping the best 5 out of the 20 candidate pool

In [ ]:
def passage_text(payload):
    parts = []
    for field in ["title", "page_content"]:
        value = payload.get(field)
        if value:
            parts.append(value)
    return "\n\n".join(parts)

pairs = [[query, passage_text(h.payload)] for h in candidates]
scores = reranker.predict(pairs, batch_size=16)

candidates_with_scores = list(zip(candidates, scores))
reranked = sorted(candidates_with_scores, key=lambda pair: pair[1], reverse=True)

print("Top-5 nach Reranking:\n")
for i, (h, s) in enumerate(reranked[:5], 1):
    print(f"[{i}] rerank={s:.4f} | {h.payload['lecture']} p.{h.payload['page_numbers']} | {h.payload['title']}")

## Before and after comparison

Printing a before/after table to show how strongly reranking reorders the hybrid results

In [ ]:
print(f"{'#':>2}  {'Hybrid (RRF)':<45}  {'→ nach Reranking':<45}")
print("-" * 96)
for i in range(5):
    h_hit = candidates[i]
    r_hit, r_score = reranked[i]
    h_str = f"{h_hit.payload['lecture']} p.{h_hit.payload['page_numbers']} {h_hit.payload['title']}"[:43]
    r_str = f"{r_hit.payload['lecture']} p.{r_hit.payload['page_numbers']} {r_hit.payload['title']}"[:43]
    moved = "" if h_hit.payload["chunk_id"] == r_hit.payload["chunk_id"] else "  <-- verschoben"
    print(f"{i+1:>2}  {h_str:<45}  {r_str:<45}{moved}")

## Full reusable retrieve() function

Wrapping hybrid retrieval plus reranking into a single retrieve() function

In [ ]:
def retrieve(query: str, top_k: int = 100, top_n: int = 10) -> list[dict]:
    candidates = hybrid_search(query, prefetch_limit=top_k)
    if not candidates:
        return []

    pairs = []
    for hit in candidates:
        pairs.append([query, passage_text(hit.payload)])
    scores = reranker.predict(pairs, batch_size=16)

    scored_candidates = []
    for hit, score in zip(candidates, scores):
        scored_candidates.append({"hit": hit, "score": float(score)})
    scored_candidates.sort(key=lambda item: item["score"], reverse=True)
    top_candidates = scored_candidates[:top_n]

    results = []
    for entry in top_candidates:
        payload = entry["hit"].payload
        results.append({
            "chunk_id": payload["chunk_id"],
            "lecture": payload["lecture"],
            "title": payload["title"],
            "page_numbers": payload["page_numbers"],
            "page_content": payload["page_content"],
            "rerank_score": entry["score"],
        })
    return results

question = """

Kannst du mir das erläutern?

Handelsklasse A
(wahre Klasse)
Handelsklasse B
(wahre Klasse)
Ausschuss
(wahre Klasse)
Handelsklasse A (prog.) 246 132 1
Handelsklasse B (prog.) 17 746 5
Ausschuss (prog.) 2 24 132


"""
results = retrieve(question)
for i, r in enumerate(results, 1):
    print(f"[{i}] {r['rerank_score']:.4f} | {r['lecture']} p.{r['page_numbers']} | {r['title']}")

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

EVAL_DIR = Path(os.getcwd()) / ".." / "data" / "eval"


def load_goldset(filename: str) -> list[dict]:
    with open(EVAL_DIR / "test_jsonfiles" / "golden" / filename, encoding="utf-8") as f:
        return json.load(f)

gold_single = load_goldset("single_source_goldset.json")
gold_multi = load_goldset("multi_source_goldset.json")
gold_all = gold_single + gold_multi


TOP_KS = [3, 5, 10]
DEPTH = max(TOP_KS)  


def rank_dense(question: str, depth: int) -> list[str]:
    hits = dense_only(question, limit=depth)
    return [h.payload["chunk_id"] for h in hits]


def rank_hybrid(question: str, depth: int) -> list[str]:
    hits = hybrid_search(question, prefetch_limit=100)
    return [h.payload["chunk_id"] for h in hits][:depth]


def rank_rerank(question: str, depth: int) -> list[str]:
    hits = retrieve(question, top_k=100, top_n=depth)
    return [h["chunk_id"] for h in hits]


STRATEGIES = {
    "dense": rank_dense,
    "hybrid (RRF)": rank_hybrid,
    "hybrid + rerank": rank_rerank,
}


rankings: dict[str, dict[str, list[str]]] = {
    name: {q["id"]: rank_fn(q["question"], DEPTH) for q in gold_all}
    for name, rank_fn in STRATEGIES.items()
}


def gold_positions(ranked_ids: list[str], gold_ids: set[str]) -> list[int]:
    position = {cid: rank for rank, cid in enumerate(ranked_ids, start=1)}
    return sorted(position[cid] for cid in gold_ids if cid in position)


def mrr_rank(positions: list[int]) -> float:
    return 1.0 / positions[0] if positions else 0.0


def score_at_k(positions: list[int], k: int, n_gold: int, partial: bool) -> float:
    found = sum(1 for pos in positions if pos <= k)
    if partial:
        return found / n_gold
    return 1.0 if found >= 1 else 0.0


def evaluate(items: list[dict], label: str) -> pd.DataFrame:
    rows = []
    for name in STRATEGIES:
        mrr = 0.0
        recall = {k: 0.0 for k in TOP_KS}
        for item in items:
            partial = item["type"] == "multi_source"
            gold_ids = {cid.strip() for cid in item["source_chunk_ids"]}
            positions = gold_positions(rankings[name][item["id"]], gold_ids)

            mrr += mrr_rank(positions)
            for k in TOP_KS:
                recall[k] += score_at_k(positions, k, len(gold_ids), partial)

        n = len(items)
        row = {"set": label, "strategy": name, "n": n, "MRR": round(mrr / n, 3)}
        row.update({f"recall@{k}": round(recall[k] / n, 3) for k in TOP_KS})
        rows.append(row)
    return pd.DataFrame(rows)

results = pd.concat(
    [
        evaluate(gold_single, "single source questions"),
        evaluate(gold_multi, "multi source questions"),
        evaluate(gold_all, "combined"),
    ],
    ignore_index=True,
)

print(results.to_string(index=False))

out_path = EVAL_DIR / "retrieval" / "retrieval_eval.csv"
results.to_csv(out_path, index=False)
print("\nsaved ->", out_path.resolve())